## Session 2: Persistence (Stateless vs Stateful)

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_openai import ChatOpenAI
from typing import TypedDict, Annotated
from dotenv import load_dotenv
import operator

In [ ]:
load_dotenv()

model = ChatOpenAI(model = "gpt-5.4-mini-2026-03-17" , temperature = 0)

In [ ]:
class ChatState(TypedDict):

    question: str
    answer: str
    history: Annotated[list[str], operator.add]

In [ ]:
def chatbot(state: ChatState):

    question = state['question']
    history = state.get('history', [])

    prompt = f'''You are a helpful assistant.
Use the conversation history if it is available.

History: {history}

User: {question}
'''

    answer = model.invoke(prompt).content

    return {
        'answer': answer,
        'history': [f'User: {question}', f'Assistant: {answer}']
    }

In [ ]:
graph = StateGraph(ChatState)

# nodes
graph.add_node('chatbot', chatbot)

# edges
graph.add_edge(START, 'chatbot')
graph.add_edge('chatbot', END)

## Part 1: Stateless Workflow

No checkpointer. After `invoke` finishes, the state is gone. The next call starts from zero.

In [ ]:
stateless_workflow = graph.compile()

stateless_workflow

In [ ]:
# first message
result1 = stateless_workflow.invoke({'question': 'Hi, my name is Suman'})
print(result1['answer'])
print(result1['history'])

In [ ]:
# second message - it will NOT remember the name
result2 = stateless_workflow.invoke({'question': 'What is my name?'})
print(result2['answer'])
print(result2['history'])

## Part 2: Stateful Workflow

Add a checkpointer and a `thread_id`. LangGraph saves the state after each run and loads it again for the same thread.

In [ ]:
checkpointer = MemorySaver()

stateful_workflow = graph.compile(checkpointer=checkpointer)

stateful_workflow

In [ ]:
config = {'configurable': {'thread_id': 'user-1'}}

In [ ]:
# first message
result1 = stateful_workflow.invoke({'question': 'Hi, my name is Suman'}, config=config)
print(result1['answer'])
print(result1['history'])

In [ ]:
# second message - it SHOULD remember the name
result2 = stateful_workflow.invoke({'question': 'What is my name?'}, config=config)
print(result2['answer'])
print(result2['history'])

## Part 3: Different thread_id means a different memory

In [ ]:
config2 = {'configurable': {'thread_id': 'user-2'}}

result3 = stateful_workflow.invoke({'question': 'Hi, my name is Rahul'}, config=config2)
print(result3['answer'])

In [ ]:
result4 = stateful_workflow.invoke({'question': 'What is my name?'}, config=config2)
print('user-2:', result4['answer'])

result5 = stateful_workflow.invoke({'question': 'What is my name?'}, config=config)
print('user-1:', result5['answer'])